[Study 01](forecast-accuracy.html) found two problems with the Valuation Model's revenue forecast. Its base case was no better than assuming revenue stays flat, and the range it drew around that base case, meant to hold 80% of outcomes, held about 14%.

Two changes to the model followed. The base case is now a blend of three forecasts: no change, the model's fading growth, and the sector's growth. The range now comes from how far forecasts like it actually missed in the past, widened for companies whose revenue has been volatile. This notebook scores both changes on the same companies, filing dates and actuals as Study 01.

Each number below is computed in the cell above it from the published results file, which [`scripts/run_forecast_rescore.py`](https://github.com/Courtizy/research/blob/main/scripts/run_forecast_rescore.py) builds from Study 01's point-in-time panels.

In [1]:
#| label: setup
#| code-summary: "Setup: load the results and the brand chart style"
import json, math, sys
from pathlib import Path
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import Markdown, HTML, display

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "brand"))
import palette                                     # Decision Models colors, validated for color-blind safety

pio.templates["dm"] = palette.plotly_template("valuation", "light")
pio.templates.default = "simple_white+dm"
SERIES = palette.series("valuation", "light", 3)
COLORS = {"blend": SERIES[0], "model": SERIES[1], "naive": SERIES[2]}
NAMES = {"blend": "After: blended base case", "model": "Before: fading growth", "naive": "No change"}

R = json.loads((ROOT / "site" / "data" / "forecast_accuracy_rescore.json").read_text())
acc = pd.DataFrame(R["accuracy"])
cov = pd.DataFrame(R["coverage"]).set_index("h")
pct = lambda x: f"{x:.0%}"
tval = lambda d: d["t"] if isinstance(d, dict) else d

## The answer

In [2]:
#| echo: false
display(Markdown(f"**{R['headline']['lead']}** {R['headline']['sub']}"))
display(HTML('<div class="kpis">' + "".join(f'<div class="kpi"><b>{k["value"]}</b><span>{k["label"]}</span></div>'
                                            for k in R["headline"]["kpis"]) + "</div>"))

**After the fixes, a range built to hold 80% of outcomes held 81% of outcomes it had not seen three years out, up from 14%; the 90% range held 93%.** The new base case misses three-year revenue by a median 17% against 20% for the old one, and is 7% more accurate than assuming no change (t = 4.7).

## What changed in the model

| | Before (Study 01) | After (engine 0.9.0 and later) |
|---|---|---|
| Base case | Growth from the company's history, fading to 2.5% over ten years | Geometric mean of no change, that fade, and the sector's median three-year growth, for five years; then the fade's rates |
| Range | A ±3 point triangular wobble in near-term growth | Past misses of the base case, as quantiles, multiplied by the company's revenue volatility plus 0.10 |

The blend has no fitted weights. Its one tuned choice, using three equal legs, was made after looking at this data, so the splits further down are the guard against having fit noise. The sector leg is the median of other companies in the same SIC division whose fiscal year ended in the same calendar year; the engine reads the same medians from a table.

In [3]:
s = R["sample"]; t = s["range_test"]
pd.Series({"Companies scored": s["companies_scored"], "Forecast–outcome pairs": s["observations"],
           "Base years": f'{s["base_years"][0]}–{s["base_years"][1]}',
           "Range test: base years from": t["first_base_year"], "Range test: pairs": t["observations"],
           "Range test: companies": t["companies"],
           "Filing dates the engine couldn't read (fixed in 0.7.1)": s["origins_engine_errors"]},
          name="").to_frame()

,
Companies scored,359
Forecast–outcome pairs,16618
Base years,2009–2025
Range test: base years from,2014
Range test: pairs,11372
Range test: companies,335
Filing dates the engine couldn't read (fixed in 0.7.1),1


## Are the ranges honest now?

Each base year from 2014 on is scored with a range built only from misses that were already known by then, the way the model would have been calibrated on the day. The old band is scored on exactly the same pairs.

In [4]:
#| label: fig-coverage
#| fig-cap: "Share of outcomes inside each range, by years out. Dashed lines mark what each range promises."
fig = go.Figure()
fig.add_bar(x=cov.index, y=cov.before.map(lambda d: d["share"]), name="Before: 80% band", marker_color=COLORS["model"])
for lvl, c in (("80", COLORS["blend"]), ("90", SERIES[2])):
    fig.add_bar(x=cov.index, y=cov.after.map(lambda d: d[lvl]["share"]), name=f"After: {lvl}% range", marker_color=c)
for y in (0.8, 0.9):
    fig.add_hline(y=y, line_dash="dash", line_color="#888")
fig.update_layout(barmode="group", xaxis_title="Years out", yaxis_tickformat=".0%", yaxis_range=[0, 1], height=360,
                  legend=dict(orientation="h", y=-0.25), margin=dict(t=10))
fig.show()

The ranges hold close to what they promise because they are much wider. That is the honest price of the old band being far too narrow:

In [5]:
#| code-summary: "How far each range reaches below and above the forecast (median)"
rows = []
for h, r in cov.iterrows():
    rows.append({"Years out": h, "Before, 80% band": f'-{pct(r.before["below"])} / +{pct(r.before["above"])}',
                 "After, 80% range": f'-{pct(r.after["80"]["below"])} / +{pct(r.after["80"]["above"])}',
                 "After, 90% range": f'-{pct(r.after["90"]["below"])} / +{pct(r.after["90"]["above"])}',
                 "Held (80 / 90 / 95)": " / ".join(pct(r.after[k]["share"]) for k in ("80", "90", "95"))})
pd.DataFrame(rows).set_index("Years out")

,"Before, 80% band","After, 80% range","After, 90% range",Held (80 / 90 / 95)
Years out,,,,
1,-2% / +2%,-12% / +17%,-20% / +26%,77% / 88% / 95%
2,-3% / +3%,-22% / +32%,-34% / +50%,79% / 90% / 95%
3,-4% / +4%,-28% / +42%,-41% / +69%,81% / 93% / 97%
4,-5% / +6%,-33% / +53%,-44% / +88%,84% / 93% / 97%
5,-6% / +7%,-35% / +63%,-47% / +98%,84% / 93% / 97%


### Does it hold in every slice?

A range can hit 80% on average by being too wide for some companies and too narrow for others. Two checks, three years out:

In [6]:
#| label: fig-slices
#| fig-cap: "Share inside the 80% range three years out, by base year (left) and by the company's revenue volatility (right)."
from plotly.subplots import make_subplots
yr, vol = pd.DataFrame(R["coverage_by_year"]), pd.DataFrame(R["coverage_by_volatility"])
fig = make_subplots(rows=1, cols=2, column_widths=[0.68, 0.32], horizontal_spacing=0.08)
fig.add_bar(x=yr.group, y=yr["80"], marker_color=COLORS["blend"], name="After: 80% range", row=1, col=1)
fig.add_bar(x=vol.group, y=vol["80"], marker_color=COLORS["blend"], showlegend=False, row=1, col=2)
fig.add_bar(x=yr.group, y=yr.before, marker_color=COLORS["model"], name="Before: 80% band", row=1, col=1)
fig.add_bar(x=vol.group, y=vol.before, marker_color=COLORS["model"], showlegend=False, row=1, col=2)
for c in (1, 2):
    fig.add_hline(y=0.8, line_dash="dash", line_color="#888", row=1, col=c)
fig.update_yaxes(tickformat=".0%", range=[0, 1])
fig.update_layout(barmode="group", height=360, legend=dict(orientation="h", y=-0.2), margin=dict(t=10))
fig.show()

In [7]:
#| echo: false
low = yr[yr["80"] < 0.75]
v = vol.set_index("group")["80"]
display(Markdown(
    f"Most base years land near 80%. The misses are " + ", ".join(f"{g} ({pct(x)})" for g, x in zip(low.group, low["80"]))
    + ": years whose next three years held a shock the earlier record didn't contain. A range built from the past "
    f"can't anticipate a new kind of shock. By volatility, steady companies' ranges held {pct(v['steady'])} and "
    f"volatile companies' {pct(v['volatile'])}, so the volatility scaling over-corrects a little: steady companies "
    "deserve a slightly wider range than they get, and volatile ones a slightly narrower one."))

Most base years land near 80%. The misses are 2014 (65%), 2020 (67%): years whose next three years held a shock the earlier record didn't contain. A range built from the past can't anticipate a new kind of shock. By volatility, steady companies' ranges held 78% and volatile companies' 85%, so the volatility scaling over-corrects a little: steady companies deserve a slightly wider range than they get, and volatile ones a slightly narrower one.

## Is the base case more accurate?

Error is the log ratio of forecast to actual (0.10 is about 10% off). Skill compares mean absolute error with the no-change forecast on the same pairs, and the *t* statistic clusters by company.

In [8]:
#| label: fig-error
#| fig-cap: "Median absolute error by years out."
fig = go.Figure()
for m in ["model", "naive", "blend"]:
    d = acc[acc.method == m]
    fig.add_scatter(x=d.h, y=d.median_abs_error, name=NAMES[m], mode="lines+markers", line_color=COLORS[m])
fig.update_layout(xaxis_title="Years out", yaxis_title="Median absolute log error", height=360,
                  legend=dict(orientation="h", y=-0.25), margin=dict(t=10))
fig.show()

In [9]:
#| code-summary: "Skill against no change, and the blend against the old base case directly"
sk = acc[acc.method.isin(["model", "blend"])].assign(
    cell=lambda d: d.apply(lambda r: f'{r.skill_vs_naive:+.0%} (t {abs(tval(r.loss_diff_vs_naive)):.1f})', axis=1))
tab = sk.pivot(index="method", columns="h", values="cell").rename(index=NAMES, columns=lambda h: f"{h} yr")
direct = pd.DataFrame(R["blend_vs_model"]).set_index("h")
tab.loc["After vs before, directly"] = [f'{g:+.0%} (t {abs(t):.1f})' for g, t in zip(direct.gain, direct.t)]
tab.rename_axis(index=None, columns=None)

,1 yr,2 yr,3 yr,4 yr,5 yr
After: blended base case,+5% (t 5.1),+5% (t 4.4),+7% (t 4.7),+8% (t 4.9),+8% (t 4.2)
Before: fading growth,+0% (t 0.0),-2% (t 1.0),-1% (t 0.6),+1% (t 0.3),+2% (t 0.5)
"After vs before, directly",+5% (t 5.9),+7% (t 7.0),+8% (t 6.4),+7% (t 4.8),+6% (t 3.7)


### Did the gain come from one period?

The blend was chosen on this data, so the test that matters is whether it helps in different conditions. The sample splits on whether the year being forecast came before 2020 or after.

In [10]:
#| code-summary: "Blend skill against no change, by period of the year being forecast"
rows = {}
for k, label in (("targets_before_2020", "Forecast year before 2020"), ("targets_2020_on", "Forecast year 2020 on")):
    a = pd.DataFrame(R["splits"][k]["accuracy"]); a = a[a.method == "blend"].set_index("h")
    rows[label] = {f"{h} yr": f'{r.skill_vs_naive:+.0%} (t {abs(tval(r.loss_diff_vs_naive)):.1f})' for h, r in a.iterrows()}
pd.DataFrame(rows).T.rename_axis(columns=None)

,1 yr,2 yr,3 yr,4 yr,5 yr
Forecast year before 2020,+7% (t 6.3),+6% (t 3.8),+5% (t 2.7),+3% (t 1.5),+3% (t 1.2)
Forecast year 2020 on,+2% (t 1.9),+4% (t 3.0),+9% (t 6.4),+13% (t 8.0),+12% (t 6.7)


In [11]:
#| echo: false
a = {k: pd.DataFrame(R["splits"][k]["accuracy"]).query("method == 'blend'").set_index("h") for k in R["splits"]}
pre, post = a["targets_before_2020"], a["targets_2020_on"]
weak = [h for h in pre.index if abs(tval(pre.loc[h, "loss_diff_vs_naive"])) < 2]
display(Markdown(
    f"Before 2020 the blend beat no change {pct(pre.loc[1, 'skill_vs_naive'])} one year out and "
    f"{pct(pre.loc[3, 'skill_vs_naive'])} three years out" + (
    f", but {' and '.join(f'{h}' for h in weak)} years out the gain is too small to tell from noise"
    if weak else "") + f". From 2020 on it beat no change by {pct(post.loc[5, 'skill_vs_naive'])} five years out. "
    "So the long-range gain leans on the years after the pandemic, when flat forecasts did especially badly. "
    "Expect a smaller edge in calm years."))

Before 2020 the blend beat no change 7% one year out and 5% three years out, but 4 and 5 years out the gain is too small to tell from noise. From 2020 on it beat no change by 12% five years out. So the long-range gain leans on the years after the pandemic, when flat forecasts did especially badly. Expect a smaller edge in calm years.

## Who gets over-forecast now?

In [12]:
#| label: fig-bias
#| fig-cap: "Mean signed error three years out by prior-growth decile. Above zero = forecast too high; bars are ±2 standard errors."
b = pd.DataFrame(R["bias_by_growth"])
fig = go.Figure()
for m in ["model", "blend"]:
    fig.add_scatter(x=b.group, y=b[m].map(lambda x: x["mean"]), name=NAMES[m], mode="lines+markers", line_color=COLORS[m],
                    error_y=dict(array=b[m].map(lambda x: 2 * x["se"]), thickness=1))
fig.add_hline(y=0, line_color="#999")
fig.update_layout(xaxis_title="Prior growth decile (1 = slowest)", yaxis_title="Mean log error", height=360,
                  legend=dict(orientation="h", y=-0.25), margin=dict(t=10))
fig.show()

In [13]:
#| echo: false
top, bot = b.iloc[-1], b.iloc[0]
display(Markdown(
    f"The fastest growers' over-forecast is gone: {top.model['mean']:+.2f} before, {top.blend['mean']:+.2f} after "
    f"(t = {top.blend['t']:.1f}). The slowest tenth, whose revenue had been shrinking a median "
    f"{pct(-bot.g3_median)} a year, is still forecast too low ({bot.blend['mean']:+.2f}, t = {bot.blend['t']:.1f}): "
    "shrinking companies tend to recover more than any of these forecasts expect."))

The fastest growers' over-forecast is gone: +0.21 before, -0.04 after (t = -0.7). The slowest tenth, whose revenue had been shrinking a median 12% a year, is still forecast too low (-0.22, t = -5.3): shrinking companies tend to recover more than any of these forecasts expect.

## By sector

In [14]:
#| code-summary: "Skill against no change three years out, sectors with at least 10 companies"
d = pd.DataFrame(R["by_division"]).set_index("division")
d.assign(**{"Before": d.model.map(lambda x: f"{x:+.0%}"), "After": d.blend.map(lambda x: f"{x:+.0%}")})[["companies", "Before", "After"]]

,companies,Before,After
division,,,
Manufacturing,140,-11%,+1%
Mining and energy,13,-15%,-9%
Retail,49,+21%,+21%
Services,38,+25%,+20%
"Transport, comms and utilities",71,-11%,+2%
Wholesale,24,+5%,+15%


## What it can't tell you

- **The blend was picked on this data.** No weights were fitted, but the choice to blend was made after seeing Study 01. The splits above are a partial check; 10-Ks filed from 2026 on are the real test, and this notebook will be re-run on them.
- **Ranges can't foresee a new kind of shock.** The 2014 and 2020 base years show it. A range built from history is only as wide as history.
- **Large US filers only.** Smaller and private companies are likely more volatile; the calibration does not cover them.
- **Revenue only.** Margins, reinvestment and the discount rate add their own uncertainty to a valuation.
- Associations, not causes. Analysis of method, not investment advice.

## Reproduce

```bash
pip install -e ".[dev]" && scripts/setup_engine.sh
export SEC_USER_AGENT="Your Name you@example.com"
python scripts/run_forecast_accuracy.py     # the point-in-time panels (Study 01)
python scripts/run_forecast_rescore.py      # this study, in seconds
```